# 01 — Run and import external comparator measurements

This notebook runs lightweight local comparators and imports outputs from
version-pinned external model environments.

It logs failures explicitly. Missing tools or failed recordings are never
encoded as clean or zero.


In [ ]:
from pathlib import Path
import pandas as pd

from paper1_qc.external_validation import (
    discover_project_root,
    load_audit_config,
    load_audio_manifest,
    run_external_comparators,
    tool_availability,
)

PROJECT_ROOT = discover_project_root()
CONFIG = load_audit_config(PROJECT_ROOT)


In [ ]:
availability = tool_availability(CONFIG)
display(availability)

manifest = load_audio_manifest(PROJECT_ROOT, CONFIG)
display(manifest.head())
print("Manifest rows:", len(manifest))
print("Missing audio paths:", int((~manifest["audio_exists"]).sum()))


For the first run, keep `run.max_recordings` small in
`config/external_validation.yaml`. After the smoke test succeeds, set it to
`null` for the full cohort.


In [ ]:
paths = run_external_comparators(PROJECT_ROOT, CONFIG)
paths


In [ ]:
measurements = pd.read_csv(paths["measurements"])
errors = pd.read_csv(paths["errors"]) if Path(paths["errors"]).stat().st_size else pd.DataFrame()

display(measurements.head())
print("Comparator rows:", len(measurements))
print("Error rows:", len(errors))
display(errors.head(20))


## Required checks

1. Verify the external tool versions and local model provenance.
2. Confirm that imported score tables use the correct recording identity.
3. Inspect errors rather than silently dropping failed recordings.
4. Do not interpret global perceptual scores as physical artifact measurements.
